#Silver Layer

In [0]:
from pyspark.sql.types import StructType, StructField,IntegerType, StringType,BooleanType, DoubleType,LongType,TimestampType
import pyspark.sql.functions as F
#all necessary libraries are imported

#simple dataframe read to see the data
df = spark.read.csv("/Volumes/workspace/accenture_final_project/lakehouse/bronze/Reseller.csv", header=True)
display(df)

**Investigating separating delimeter**

In [0]:

df = spark.read.text("/Volumes/workspace/accenture_final_project/lakehouse/bronze/Reseller.csv")
df = df.withColumnRenamed("value", "raw_data")
df.show(5, truncate=False)

**Reading the data with delimeter tab (\t) as a dataframe**

In [0]:

df = spark.read.csv("/Volumes/workspace/accenture_final_project/lakehouse/bronze/Reseller.csv", sep="\t", header=True)
df.show(5, truncate=False)

In [0]:
# define the schema
schema = StructType([
    StructField("reseller_key",IntegerType(), True),
    StructField("business_type",StringType(), True),
    StructField("reseller",StringType(), True),
    StructField("city",StringType(), True),
    StructField("state_province",StringType(), True),
    StructField("country_region",StringType(),True)

])

df = spark.read.schema(schema).csv("/Volumes/workspace/accenture_final_project/lakehouse/bronze/Reseller.csv", sep="\t",header=True)
display(df)

##Cleaning the data

In [0]:
df_silver = (
    df.dropDuplicates()
    #1.removing whitespace
    .withColumn("business_type", F.trim(F.col("business_type")))
    .withColumn("reseller", F.trim(F.col("reseller")))
    .withColumn("city", F.trim(F.col("city")))
    .withColumn("state_province", F.trim(F.col("state_province")))
    .withColumn("country_region", F.trim(F.col("country_region")))

)
# Εμφάνιση του αποτελέσματος στο Databricks
display(df_silver)

In [0]:
%sql
DROP TABLE IF EXISTS accenture_final_project.silver_reseller;

##Save the data into a Delta table

In [0]:

df_silver.write.mode("overwrite").format("delta").saveAsTable("accenture_final_project.silver_Reseller")